In [36]:
import os
import dotenv
import pandas as pd
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError

dotenv.load_dotenv()

api_key = os.getenv("ENTSO_API_KEY")

In [37]:
client = EntsoePandasClient(api_key=api_key)

start = pd.Timestamp("2024-10-01", tz="Europe/Brussels")
end = pd.Timestamp("2026-10-01", tz="Europe/Brussels")


#codes of zones which prices are affecting NO2
price_codes = ["NL", "DE_LU", "BE", "FR"] #BE belgium or FR france DE_LU germany luxembourg
neighbor_codes = ["NO_5", "DK_1", "NO_1", "NL", "DE_LU"]


In [38]:
country_prices = []
for code in price_codes:
    print(code)
    prices = client.query_day_ahead_prices(
        country_code=code,
        start=start,
        end=end
    )
    # query_day_ahead_prices returns a Series named "Price";
    # rename it to "price_<zone>" so concat produces unique columns
    country_prices.append(prices.rename(f"price_{code}"))
# sort=True + sort_index(): all zones publish hourly day-ahead prices here,
# but concatenating indexes from different timezones can leave the union
# unsorted. An unsorted index silently corrupts any later ffill, because
# ffill walks rows in order rather than in time order.
country_prices_df = pd.concat(country_prices, axis="columns", sort=True).sort_index()

NL
DE_LU
BE
FR


In [39]:
def query_wind_solar_gen_and_load_forecast(country_code, start, end):
    code = country_code.lower()
    wind_solar_forecast = client.query_wind_and_solar_forecast(
        country_code=country_code,
        start=start,
        end=end
    )
    generation_forecast = client.query_generation_forecast(
        country_code=country_code,
        start=start,
        end=end
    )
    load_forecast = client.query_load_forecast(
        country_code=country_code,
        start=start,
        end=end
    )
    # prefix every column with the country code so concat produces unique columns
    wind_solar_forecast.columns = [
        f"{code}_{column.strip().lower().replace(' ', '_')}"
        for column in wind_solar_forecast.columns
    ]
    # entsoe-py returns a Series whose name ends with "- Actual Aggregated";
    # relabel it as a forecast since document A75 contains day-ahead
    generation_forecast.name = f"{code}_forecasted_generation"
    # NOTE: query_load_forecast returns a DataFrame whose single column is
    # always literally named "Forecasted Load". Assigning .name on a DataFrame
    # is a no-op (it just sets an ad-hoc attribute), which would leave every
    # zone with an identically named column and lose the zone information on
    # concat. Rename the columns instead.
    load_forecast = load_forecast.rename(
        columns=lambda column: f"{code}_forecasted_load"
    )
    return wind_solar_forecast, generation_forecast, load_forecast


In [40]:
# A69/A75/A65: wind+solar, generation and load forecasts for NO_2 and its neighbors
forecast_codes = ["NO_2", "NO_1", "NO_5", "DE_LU"]

wind_solar_forecasts = []
generation_forecasts = []
load_forecasts = []
for code in forecast_codes:
    print(code)
    wind_solar_forecast, generation_forecast, load_forecast = query_wind_solar_gen_and_load_forecast(
        country_code=code,
        start=start,
        end=end
    )
    wind_solar_forecasts.append(wind_solar_forecast)
    generation_forecasts.append(generation_forecast)
    load_forecasts.append(load_forecast)

wind_solar_forecasts_df = pd.concat(wind_solar_forecasts, axis="columns", sort=True).sort_index()
generation_forecasts_df = pd.concat(generation_forecasts, axis="columns", sort=True).sort_index()
load_forecasts_df = pd.concat(load_forecasts, axis="columns", sort=True).sort_index()

# A72 -> returns a pandas Series
reservoir_codes = ["NO_2", "NO_1", "NO_5"]
reservoirs = []
for code in reservoir_codes:
    print(code)
    reservoir = client.query_aggregate_water_reservoirs_and_hydro_storage(
        country_code=code,
        start=start,
        end=end
    )
    reservoirs.append(reservoir.rename(f"{code.lower()}_reservoir_level_MWh"))
reservoirs_df = pd.concat(reservoirs, axis="columns", sort=True).sort_index()

def describe_resolution(df, label):
    """Report the sampling interval of a frame and average it to hourly.

    ENTSO-E changed resolution twice, and not on the same date for every
    document:
      * forecasts (A69/A71/A65) went 60min -> 15min around 2025-03-19
      * day-ahead prices (A44) went 60min -> 15min on 2025-10-01
    DE_LU already published 15min forecasts before this window opened.
    So a frame can be a mix of hourly and quarter-hourly rows at the same
    time, and that mix changes partway through the series.

    Averaging is the right collapse for volume-like quantities (the sum of
    the four quarters is the hour total), but it does change what a price
    column means, so we always report when it happened.
    """
    if len(df.index) < 2:
        print(f"{label}: fewer than 2 rows, nothing to collapse")
        return df.resample("h").mean()
    was_sorted = df.index.is_monotonic_increasing
    if not was_sorted:
        print(
            f"{label}: index was NOT sorted - sorting before averaging, "
            f"otherwise ffill/resample would mix up timestamps"
        )
        df = df.sort_index()
    steps = df.index.to_series().diff().dropna()
    median_step = steps.median()
    sub_hour = int((steps < pd.Timedelta("1h")).sum())
    hours = median_step / pd.Timedelta("1h")
    print(
        f"{label}: {len(df)} rows, median step {median_step} "
        f"(~{hours:.2f}h), {sub_hour} sub-hourly steps"
    )
    if median_step < pd.Timedelta("1h"):
        print(
            f"    -> sub-hourly source ({median_step}): "
            f"averaging quarters to hourly"
        )
    else:
        print("    -> already hourly, avg is a no-op")
    hourly = df.resample("h").mean()
    print(
        f"    -> {len(df)} rows in, {len(hourly)} rows out "
        f"({len(df) - len(hourly)} collapsed)"
    )
    return hourly

forecasts_df = pd.concat(
    [wind_solar_forecasts_df, generation_forecasts_df, load_forecasts_df],
    axis="columns",
    sort=True,
).sort_index()
forecasts_hourly = describe_resolution(forecasts_df, "forecasts")

print()
prices_hourly = describe_resolution(country_prices_df, "day-ahead prices")

# NOTE on prices: entsoe-py already forces 60min resolution before the
# 2025-10-01 SDAC MTU change, so pre-2025-10 rows are true hourly prices.
# After that date they are 15min prices, so the price column becomes an
# hour-average rather than a settled hourly price. That comes from the
# source data rather than from anything we do here, but it does mean the
# price column means slightly different things in the three regimes:
#   2024-10 -> 2025-03  hourly forecasts, hourly prices  (true hourly values)
#   2025-03 -> 2025-10  15min forecasts, hourly prices   (forecasts averaged)
#   2025-10 -> present  15min forecasts, 15min prices   (both averaged)
combined_df = prices_hourly.join(forecasts_hourly, how="outer").sort_index()

for column in reservoirs_df.columns:
    combined_df[column] = reservoirs_df[column].reindex(
        combined_df.index,
        method="ffill"
    )


NO_2
NO_1
NO_5
DE_LU
NO_2
NO_1
NO_5
forecasts: 70080 rows, median step 0 days 00:15:00 (~0.25h), 70079 sub-hourly steps
    -> sub-hourly source (0 days 00:15:00): averaging quarters to hourly
    -> 70080 rows in, 17520 rows out (52560 collapsed)

day-ahead prices: 43799 rows, median step 0 days 00:15:00 (~0.25h), 35039 sub-hourly steps
    -> sub-hourly source (0 days 00:15:00): averaging quarters to hourly
    -> 43799 rows in, 17521 rows out (26278 collapsed)


In [41]:
print(combined_df.head())

# combined_df is already hourly here: the forecasts were averaged down to
# hourly in the previous cell and the prices are hourly to begin with.
# Resampling again here would be a no-op at best. ffill is safe because the
# index is sorted, so it walks in time order and only fills interior gaps
# (leading NaNs are intentionally left alone by ffill).
combined_df = combined_df.sort_index().ffill()

combined_df.to_csv("entsoe_data_hourly.csv")

                           price_NL  price_DE_LU  price_BE  price_FR  \
2024-09-30 22:00:00+00:00      3.21         3.21      3.21      3.21   
2024-09-30 23:00:00+00:00      0.07         0.07      0.07      0.07   
2024-10-01 00:00:00+00:00      0.05         0.05      0.05      0.05   
2024-10-01 01:00:00+00:00      0.02         0.02      0.02      0.02   
2024-10-01 02:00:00+00:00      0.09         0.09      0.09      0.09   

                           no_2_solar  no_2_wind_onshore  no_2_wind_offshore  \
2024-09-30 22:00:00+00:00         0.0             949.94                 NaN   
2024-09-30 23:00:00+00:00         0.0             926.94                 NaN   
2024-10-01 00:00:00+00:00         0.0             897.66                 NaN   
2024-10-01 01:00:00+00:00         0.0             866.61                 NaN   
2024-10-01 02:00:00+00:00         0.0             837.23                 NaN   

                           no_1_solar  no_1_wind_onshore  no_5_solar  ...  \
2024-09-3